# 租金合适

多代理系统扫描**纽约、拉各斯和内罗毕**的租赁列表，使用人工智能模型集合估算公平的市场租金，并提醒用户定价过低的交易。

## 涵盖的概念
- Modal.com 无服务器 GPU 部署（SpecialistAgent）
- 带色度矢量数据库的 RAG (FrontierAgent)
- 多个模型的加权集成
- OpenAI 结构化输出 (ScannerAgent)
- OpenAI函数调用/代理循环（AutonomousAgent）
- 通过 Pushover API (MessagingAgent) 推送通知
- 具有 3D 可视化效果的渐变交互式仪表板

## 设置

在此目录中创建一个 `.env` 文件：
```
OPENAI_API_KEY=your_key
PUSHOVER_TOKEN=your_token (optional)
PUSHOVER_USER=your_user (optional)
```

In [ ]:
!uv pip install openai chromadb sentence-transformers gradio plotly scikit-learn modal pydantic requests python-dotenv datasets

In [ ]:
import sys
sys.path.append('.')

from dotenv import load_dotenv
load_dotenv()

## 步骤 1：数据模型和模拟列表

我们为代理之间的类型安全数据流定义了 Pydantic 模型，并为我们的三个目标城市生成了真实的租赁列表的模拟提要。

In [ ]:
from agents.rental_deals import (
    ScrapedListing, RentalDeal, DealSelection, RentalOpportunity,
    generate_mock_listings, fetch_all_listings, scraped_to_deal
)

# Generate sample listings for each city
for city in ["New York", "Lagos", "Nairobi"]:
    listings = generate_mock_listings(city, count=3)
    print(f"\n--- {city} ---")
    for l in listings:
        print(f"  {l.title} | ${l.price:,.2f}/mo")

## 步骤 2：ScannerAgent — 使用结构化输出进行交易发现

ScannerAgent 获取所有列表并要求 GPT-4o-mini 选择 5 个最佳交易。它使用 OpenAI 结构化输出（`response_format=DealSelection`），因此响应是经过验证的 Pydantic 对象 - 无需手动 JSON 解析。

In [ ]:
from agents.rental_scanner_agent import RentalScannerAgent

scanner = RentalScannerAgent()
deals = scanner.scan(count_per_city=5)

for d in deals:
    print(f"{d.title} | {d.bedrooms}BR | {d.sqft}sqft | ${d.rent:,.2f}/mo")

## 步骤 3：获取数据集并设置矢量数据库

我们从 HuggingFace 获取精心策划的租赁数据集，然后使用“all-MiniLM-L6-v2”将列表嵌入到 Chroma 矢量数据库中。这就是 FrontierAgent 查询 RAG 上下文的内容。

In [ ]:
from datasets import load_dataset
import chromadb
from sentence_transformers import SentenceTransformer

# Fetch curated rental dataset from HuggingFace
dataset = load_dataset("Gasmyr/rental-prices", split="train")
print(f"Loaded {len(dataset)} rental listings from HuggingFace.")

# Set up embedding model and Chroma
model = SentenceTransformer("all-MiniLM-L6-v2")
client = chromadb.PersistentClient(path="rental_vectorstore")
collection = client.get_or_create_collection("rental_listings")

# Reset if vector store is out of sync with dataset
if collection.count() != len(dataset):
    print(f"Vector DB has {collection.count()} items but dataset has {len(dataset)}. Resetting...")
    client.delete_collection("rental_listings")
    collection = client.get_or_create_collection("rental_listings")

# Populate vector DB with dataset (batch encoding for speed)
if collection.count() == 0:
    texts = [
        f"{row['bedrooms']}BR {row['sqft']}sqft in {row['city']}. {row['description']}"
        for row in dataset
    ]
    ids = [f"listing_{i}" for i in range(len(dataset))]
    metadatas = [
        {"city": row["city"], "rent": float(row["rent"]), "bedrooms": row["bedrooms"]}
        for row in dataset
    ]

    BATCH_SIZE = 500
    for start in range(0, len(texts), BATCH_SIZE):
        end = min(start + BATCH_SIZE, len(texts))
        batch_texts = texts[start:end]
        batch_embeddings = model.encode(batch_texts, show_progress_bar=False).tolist()
        collection.add(
            ids=ids[start:end],
            embeddings=batch_embeddings,
            documents=batch_texts,
            metadatas=metadatas[start:end],
        )
        print(f"  Embedded {end}/{len(texts)} listings...")

    print(f"Added {collection.count()} listings to vector DB.")
else:
    print(f"Vector DB already has {collection.count()} listings.")

## 步骤 4：FrontierAgent — 基于 RAG 的租金估算

FrontierAgent 对目标属性进行编码，从 Chroma 检索 5 个最相似的列表，并将它们作为上下文发送到 GPT-4o。该模型使用这些可比较的房产来估计公平租金。

In [ ]:
from agents.rental_frontier_agent import RentalFrontierAgent

frontier = RentalFrontierAgent(collection=collection)

# Estimate fair rent for the first scanned deal
if deals:
    test_deal = deals[0]
    estimate = frontier.estimate(test_deal)
    print(f"\nListed: ${test_deal.rent:,.2f}/mo")
    print(f"Estimated fair rent: ${estimate:,.2f}/mo")
    print(f"Difference: ${estimate - test_deal.rent:,.2f}/mo")

## 步骤 4b：模态上的 SpecialistAgent

微调后的模型权重已发布在 HuggingFace 上的“Gasmyr/rental-pricer”中。 **您不需要重新训练。** 您只需要在自己的 Modal 帐户上部署推理服务。

**首次设置：**
```bash
modal setup                  # one-time authentication
modal secret create huggingface-secret HF_TOKEN=your_hf_token
```

**部署推理服务（使用预先训练的权重）：**
```bash
modal deploy rental_pricer_service.py
```

**仅当您想从头开始重新训练时（约 1-3 小时，约 2-5 美元）：**
```bash
modal run rental_pricer_service.py::train
```

如果未部署 Modal，集成将优雅地回退到 Frontier 代理。

In [ ]:
# Optional: test the Modal service directly before using it in the ensemble
import modal

try:
    estimate_rent = modal.Function.from_name("rental-pricer-service", "estimate_rent")
    test_result = estimate_rent.remote("2-bedroom, 850 sqft in New York. Modern kitchen, near public transit.")
    print(f"Modal specialist estimate: ${float(test_result):,.2f}/mo")
except Exception as e:
    print(f"Modal not deployed yet: {e}\nRun 'modal deploy rental_pricer_service.py' first, or skip — ensemble will fallback.")

## 步骤5：EnsembleAgent——加权模型组合

EnsembleAgent 结合了三个模型：
- **80%** FrontierAgent (RAG + GPT) — 最准确
- **10%** SpecialistAgent（在 Modal 上微调 Llama）
- **10%** 神经网络

注意：SpecialistAgent 需要 Modal 部署。对于本地测试，整体会优雅地回落。

In [ ]:
from agents.rental_ensemble_agent import RentalEnsembleAgent
from agents.rental_specialist_agent import RentalSpecialistAgent

# SpecialistAgent requires Modal deployment — set to None if not deployed yet
try:
    specialist = RentalSpecialistAgent()
except Exception as e:
    print(f"Modal not available ({e}). Ensemble will fallback to Frontier for specialist weight.")
    specialist = None

ensemble = RentalEnsembleAgent(
    frontier=frontier,
    specialist=specialist,
)

if deals:
    test_deal = deals[0]
    estimate = ensemble.estimate(test_deal)
    print(f"\nEnsemble estimate: ${estimate:,.2f}/mo")

## 步骤 6：MessagingAgent — 推送通知

MessagingAgent 使用 Claude Sonnet 制作引人注目的警报，然后通过 Pushover 发送。如果未设置 Pushover 凭据，则会记录消息。

In [ ]:
from agents.rental_messaging_agent import RentalMessagingAgent
from agents.rental_deals import RentalOpportunity

messenger = RentalMessagingAgent()

if deals:
    test_opp = RentalOpportunity(
        deal=deals[0],
        estimated_fair_rent=estimate,
        monthly_savings=max(estimate - deals[0].rent, 0),
    )
    messenger.alert(test_opp)

## 步骤 7：AutonomousAgent — 使用工具的代理循环

AutonomousAgent 使用 OpenAI 函数调用进行自我引导。它定义了 3 个工具（扫描、估计、警报），GPT 通过代理 while 循环自主决定执行顺序。

In [ ]:
from agents.rental_autonomous_agent import RentalAutonomousAgent

autonomous = RentalAutonomousAgent(
    scanner=scanner,
    ensemble=ensemble,
    messenger=messenger,
)

opportunities = autonomous.run()

print(f"\n=== Found {len(opportunities)} opportunities ===")
for opp in sorted(opportunities, key=lambda o: o.monthly_savings, reverse=True):
    print(f"{opp.deal.title} | Listed: ${opp.deal.rent:,.0f} | Fair: ${opp.estimated_fair_rent:,.0f} | Savings: ${opp.monthly_savings:,.0f}")

## 步骤 8：3D 矢量可视化

以 3D 形式可视化色度矢量数据库。列表按城市聚集，确认嵌入捕获地理和定价模式。

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn.decomposition import PCA

result = collection.get(include=["embeddings", "metadatas", "documents"])

if result["embeddings"] is not None and len(result["embeddings"]) > 0:
    embeddings = np.array(result["embeddings"])
    coords = PCA(n_components=3).fit_transform(embeddings)

    city_colors = {"New York": "red", "Lagos": "green", "Nairobi": "blue"}
    colors = [city_colors.get(m.get("city", ""), "gray") for m in result["metadatas"]]
    labels = [d[:60] for d in result["documents"]]

    fig = go.Figure(data=[go.Scatter3d(
        x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
        mode="markers",
        marker=dict(size=3, color=colors, opacity=0.7),
        text=labels, hoverinfo="text",
    )])
    fig.update_layout(title="Rental Listings Vector Space", height=500)
    fig.show()
else:
    print("No embeddings in vector DB yet.")

## 步骤 9：启动 Gradio 仪表板

运行完整的交互式 UI，包括交易搜寻、警报和矢量可视化。

In [ ]:
from the_rent_is_right import create_ui

demo = create_ui()
demo.launch()